<a href="https://colab.research.google.com/github/asenin592/Programacion-de-Sistemas-de-Base-1/blob/main/Unidad%203/Collab/GONZ%C3%81LEZ%20FRANCISCO%20PSSB%20I%20Colab%20U3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Programación de Sistemas de Base I - Expresiones Regulares (ER) y Autómatas Finitos
**Asignatura:** Programación de Sistemas de Base I (8.º Semestre)  
**Unidad III:** Análisis Léxico  
**Tiempo estimado:** 3 horas (Asíncrono / Guiado)  
**Repositorio Base / Entrega:** Integración con Moodle / GitHub

---


## 1. Objetivos de Aprendizaje
1. Implementar una clase orientada a objetos que construya la matriz de transición y ejecuta la simulación paso a paso de cadenas de entrada.
2.  Implementar una clase orientada a objetos con algoritmos explícitos para el cálculo de la cerradura epsilon, la función move y la simulación paso a paso del algoritmo.
3. Implementar una arquitectura orientada a objetos para construir autómatas finitos usando el Algoritmo de Thompson para obtener el AFN-ɛ a partir de la ER.
4. Implementar el Algoritmo de Construcción de Subconjuntos utilizando programación orientada a objetos para convertir un AFN en un AFD.

---


## 2. Sección Práctica 1: Matriz de Transición
En esta sección analizaremos el algoritmo simulador para reconocimiento de cadenas mediante un AFD y la matriz de transición.


### [CÓDIGO 1.1] AFDeterminista.py
```python
# ==============================================================================
# Tema: Emulador de Autómata Finito Determinista (AFD) y Matriz de Transición
# ==============================================================================

from typing import Set, Dict, Tuple, List, Optional

class AutomataFinitoDeterminista:
    """Clase orientada a objetos para modelar y simular un AFD (5-tupla)."""

    def __init__(self, states: Set[int], alphabet: Set[str],
                 transitions: Dict[Tuple[int, str], int],
                 initial_state: int, accepting_states: Set[int]):
        self.S = states
        self.Sigma = alphabet
        self.delta = transitions  # Corregido: atributo asignado correctamente
        self.s0 = initial_state
        self.F = accepting_states
        
        # Estado de Error implícito
        self.ERROR_STATE = -1

    def transition(self, state: int, symbol: str) -> int:
        """Consulta la función/matriz de transición delta(s, c)."""
        return self.delta.get((state, symbol), self.ERROR_STATE) # Corregido: llamado al atributo delta

    def evaluate_string(self, input_string: str) -> Tuple[bool, List[str]]:
        """Simula la ejecución del AFD carácter a carácter trazando los estados."""
        current_state = self.s0
        trace = [f"Estado Inicial: q{current_state}"]

        for i, char in enumerate(input_string):
            if char not in self.Sigma:
                trace.append(f"  [Error] Carácter '{char}' fuera del alfabeto Sigma.")
                return False, trace

            next_state = self.transition(current_state, char)
            if next_state == self.ERROR_STATE:
                trace.append(f"  [Bloqueo] Sin transición desde q{current_state} con '{char}'.")
                return False, trace

            trace.append(f"  Paso {i+1}: Símbolo '{char}' | Transición q{current_state} -> q{next_state}")
            current_state = next_state

        is_accepted = current_state in self.F
        status = "CADENA ACEPTADA" if is_accepted else "CADENA RECHAZADA"
        trace.append(f"Estado Final: q{current_state} | Resultado: {status}")
        return is_accepted, trace

    def print_transition_matrix(self):
        """Imprime la representación tabular/matriz de transición del AFD."""
        sorted_symbols = sorted(list(self.Sigma))
        sorted_states = sorted(list(self.S))

        print("\n=== MATRIZ DE TRANSICIÓN Dtran[s, c] ===")
        header = f"{'Estado':<8} | " + " | ".join([f"{sym:^5}" for sym in sorted_symbols])
        print(header)
        print("-" * len(header))

        for state in sorted_states:
            prefix = "-> " if state == self.s0 else (" * " if state in self.F else "   ")
            row_str = f"{prefix}q{state:<4} | "
            row_cells = []
            for sym in sorted_symbols:
                dest = self.transition(state, sym)
                cell = f"q{dest}" if dest != self.ERROR_STATE else "--"
                row_cells.append(f"{cell:^5}")
            row_str += " | ".join(row_cells)
            print(row_str)

# --- DEMOSTRACIÓN PRÁCTICA Y EVALUACIÓN ---
if __name__ == "__main__":
    # Construcción de un AFD para reconocer identificadores simples: Letra (L) seguida de Letras (L) o Dígitos (D)
    # Expresión regular equivalente: L (L | D)*
    
    states_set = {0, 1}
    alphabet_set = {'L', 'D'}
    
    # Transiciones:
    # q0 --L--> q1
    # q1 --L--> q1
    # q1 --D--> q1
    transitions_map = {
        (0, 'L'): 1,
        (1, 'L'): 1,
        (1, 'D'): 1
    }
    
    initial = 0
    accepting = {1}

    afd_id = AutomataFinitoDeterminista(states_set, alphabet_set, transitions_map, initial, accepting)

    # Mostrar la matriz de transición
    afd_id.print_transition_matrix()

    # Batería de pruebas
    test_inputs = ["LLD", "L1D", "DLL", "L"]
    
    print("\n=== EVALUACIÓN DE CADENAS DE PRUEBA ===")
    for test in test_inputs:
        # Convertir caracteres para simplificar la prueba (letras -> 'L', dígitos -> 'D')
        mapped_input = "".join(['L' if c.isalpha() else ('D' if c.isdigit() else c) for c in test])
        print(f"\nEvaluating Input Original: '{test}' -> Abstract: '{mapped_input}'")
        accepted, execution_trace = afd_id.evaluate_string(mapped_input)
        for line in execution_trace:
            print(line)
```



---

In [ ]:
from typing import Set, Dict, Tuple, List, Optional

class AutomataFinitoDeterminista:
    """Clase orientada a objetos para modelar y simular un AFD (5-tupla)."""

    def __init__(self, states: Set[int], alphabet: Set[str],
                 transitions: Dict[Tuple[int, str], int],
                 initial_state: int, accepting_states: Set[int]):
        self.S = states
        self.Sigma = alphabet
        self.delta = transitions  # Corregido: atributo asignado correctamente
        self.s0 = initial_state
        self.F = accepting_states

        # Estado de Error implícito
        self.ERROR_STATE = -1

    def transition(self, state: int, symbol: str) -> int:
        """Consulta la función/matriz de transición delta(s, c)."""
        return self.delta.get((state, symbol), self.ERROR_STATE) # Corregido: llamado al atributo delta

    def evaluate_string(self, input_string: str) -> Tuple[bool, List[str]]:
        """Simula la ejecución del AFD carácter a carácter trazando los estados."""
        current_state = self.s0
        trace = [f"Estado Inicial: q{current_state}"]

        for i, char in enumerate(input_string):
            if char not in self.Sigma:
                trace.append(f"  [Error] Carácter '{char}' fuera del alfabeto Sigma.")
                return False, trace

            next_state = self.transition(current_state, char)
            if next_state == self.ERROR_STATE:
                trace.append(f"  [Bloqueo] Sin transición desde q{current_state} con '{char}'.")
                return False, trace

            trace.append(f"  Paso {i+1}: Símbolo '{char}' | Transición q{current_state} -> q{next_state}")
            current_state = next_state

        is_accepted = current_state in self.F
        status = "CADENA ACEPTADA" if is_accepted else "CADENA RECHAZADA"
        trace.append(f"Estado Final: q{current_state} | Resultado: {status}")
        return is_accepted, trace

    def print_transition_matrix(self):
        """Imprime la representación tabular/matriz de transición del AFD."""
        sorted_symbols = sorted(list(self.Sigma))
        sorted_states = sorted(list(self.S))

        print("\n=== MATRIZ DE TRANSICIÓN Dtran[s, c] ===")
        header = f"{'Estado':<8} | " + " | ".join([f"{sym:^5}" for sym in sorted_symbols])
        print(header)
        print("-" * len(header))

        for state in sorted_states:
            prefix = "-> " if state == self.s0 else (" * " if state in self.F else "   ")
            row_str = f"{prefix}q{state:<4} | "
            row_cells = []
            for sym in sorted_symbols:
                dest = self.transition(state, sym)
                cell = f"q{dest}" if dest != self.ERROR_STATE else "--"
                row_cells.append(f"{cell:^5}")
            row_str += " | ".join(row_cells)
            print(row_str)

# --- DEMOSTRACIÓN PRÁCTICA Y EVALUACIÓN ---
if __name__ == "__main__":
    # Construcción de un AFD para reconocer identificadores simples: Letra (L) seguida de Letras (L) o Dígitos (D)
    # Expresión regular equivalente: L (L | D)*

    states_set = {0, 1}
    alphabet_set = {'L', 'D'}

    # Transiciones:
    # q0 --L--> q1
    # q1 --L--> q1
    # q1 --D--> q1
    transitions_map = {
        (0, 'L'): 1,
        (1, 'L'): 1,
        (1, 'D'): 1
    }

    initial = 0
    accepting = {1}

    afd_id = AutomataFinitoDeterminista(states_set, alphabet_set, transitions_map, initial, accepting)

    # Mostrar la matriz de transición
    afd_id.print_transition_matrix()

    # Batería de pruebas
    test_inputs = ["LLD", "L1D", "DLL", "L"]

    print("\n=== EVALUACIÓN DE CADENAS DE PRUEBA ===")
    for test in test_inputs:
        # Convertir caracteres para simplificar la prueba (letras -> 'L', dígitos -> 'D')
        mapped_input = "".join(['L' if c.isalpha() else ('D' if c.isdigit() else c) for c in test])
        print(f"\nEvaluating Input Original: '{test}' -> Abstract: '{mapped_input}'")
        accepted, execution_trace = afd_id.evaluate_string(mapped_input)
        for line in execution_trace:
            print(line)


=== MATRIZ DE TRANSICIÓN Dtran[s, c] ===
Estado   |   D   |   L  
------------------------
-> q0    |  --   |  q1  
 * q1    |  q1   |  q1  

=== EVALUACIÓN DE CADENAS DE PRUEBA ===

Evaluating Input Original: 'LLD' -> Abstract: 'LLL'
Estado Inicial: q0
  Paso 1: Símbolo 'L' | Transición q0 -> q1
  Paso 2: Símbolo 'L' | Transición q1 -> q1
  Paso 3: Símbolo 'L' | Transición q1 -> q1
Estado Final: q1 | Resultado: CADENA ACEPTADA

Evaluating Input Original: 'L1D' -> Abstract: 'LDL'
Estado Inicial: q0
  Paso 1: Símbolo 'L' | Transición q0 -> q1
  Paso 2: Símbolo 'D' | Transición q1 -> q1
  Paso 3: Símbolo 'L' | Transición q1 -> q1
Estado Final: q1 | Resultado: CADENA ACEPTADA

Evaluating Input Original: 'DLL' -> Abstract: 'LLL'
Estado Inicial: q0
  Paso 1: Símbolo 'L' | Transición q0 -> q1
  Paso 2: Símbolo 'L' | Transición q1 -> q1
  Paso 3: Símbolo 'L' | Transición q1 -> q1
Estado Final: q1 | Resultado: CADENA ACEPTADA

Evaluating Input Original: 'L' -> Abstract: 'L'
Estado Inicial: q0

## 3. Sección Práctica 2: Cerradura-ɛ
En esta sección construiremos una clase orientada a objetos con algoritmos explícitos para el cálculo de la cerradura epsilon y la función move.

### [CÓDIGO 2.1] epsilonClosure.py
```python
# ==============================================================================
# Tema: Emulador de Autómata Finito No Determinista con Transiciones Epsilon (AFN-e)
# ==============================================================================

from typing import Set, Dict, Tuple, List, Any

class NFA_Epsilon:
    """Clase orientada a objetos para modelar y simular un AFN con transiciones epsilon."""
    EPSILON = 'ε'  # Cadena vacía constante

    def __init__(self, states: Set[int], alphabet: Set[str],
                 transitions: Dict[Tuple[int, str], Set[int]],
                 initial_state: int, accepting_states: Set[int]):
        self.S = states
        self.Sigma = alphabet
        self.delta = transitions
        self.s0 = initial_state
        self.F = accepting_states

    def epsilon_closure_state(self, state: int) -> Set[int]:
        """Calcula la epsilon-cerradura para un solo estado usando un algoritmo de pila (DFS)."""
        closure = {state}
        stack = [state]

        while stack:
            current = stack.pop()
            # Buscar transiciones epsilon desde el estado actual
            next_states = self.delta.get((current, self.EPSILON), set())
            for nxt in next_states:
                if nxt not in closure:
                    closure.add(nxt)
                    stack.append(nxt)
        return closure

    def epsilon_closure_set(self, states_set: Set[int]) -> Set[int]:
        """Calcula la epsilon-cerradura para un conjunto de estados: e-closure(T) = U e-closure(s)."""
        total_closure = set()
        for state in states_set:
            total_closure.update(self.epsilon_closure_state(state))
        return total_closure

    def move(self, states_set: Set[int], symbol: str) -> Set[int]:
        """Calcula el conjunto mover(T, a): estados alcanzables con el símbolo 'a'."""
        reachable = set()
        for state in states_set:
            nxt = self.delta.get((state, symbol), set())
            reachable.update(nxt)
        return reachable

    def simulate(self, input_string: str) -> Tuple[bool, List[str]]:
        """Simula la ejecución del AFN (Algoritmo 3.22 del Dragon Book) sobre una cadena."""
        trace = []
        # Paso 1: Estado inicial activo S = e-closure(s0)
        current_active = self.epsilon_closure_state(self.s0)
        trace.append(f"Estado Activo Inicial e-closure({self.s0}): {sorted(list(current_active))}")

        # Bucle principal sobre cada carácter de la entrada
        for i, char in enumerate(input_string):
            if char not in self.Sigma:
                trace.append(f"  [Error] Carácter '{char}' fuera del alfabeto.")
                return False, trace

            # Paso 2a: T = mover(S, c)
            move_set = self.move(current_active, char)
            # Paso 2b: S' = e-closure(T)
            next_active = self.epsilon_closure_set(move_set)

            trace.append(f"  Paso {i+1} ('{char}'): mover -> {sorted(list(move_set))} | e-closure -> {sorted(list(next_active))}")
            current_active = next_active

            if not current_active:
                trace.append("  [Bloqueo] El conjunto de estados activos quedó vacío.")
                return False, trace

        # Criterio de Aceptación: S ∩ F != Ø
        is_accepted = bool(current_active.intersection(self.F))
        status = "CADENA ACEPTADA" if is_accepted else "CADENA RECHAZADA"
        trace.append(f"Estados Activos Finales: {sorted(list(current_active))} | Intersección F: {sorted(list(current_active.intersection(self.F)))}")
        trace.append(f"Resultado: {status}")

        return is_accepted, trace

# --- DEMOSTRACIÓN PRÁCTICA Y EVALUACIÓN ---
if __name__ == "__main__":
    # Construcción del AFN-e para el lenguaje (a|b)*abb
    states_all = {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10}
    alphabet_in = {'a', 'b'}
    
    # Transiciones del AFN-e (basado en la figura 3.34 de Aho et al.)
    transitions_nfa = {
        (0, NFA_Epsilon.EPSILON): {1, 7},
        (1, NFA_Epsilon.EPSILON): {2, 4},
        (2, 'a'): {3},
        (3, NFA_Epsilon.EPSILON): {6},
        (4, 'b'): {5},
        (5, NFA_Epsilon.EPSILON): {6},
        (6, NFA_Epsilon.EPSILON): {1, 7},
        (7, 'a'): {8},
        (8, 'b'): {9},
        (9, 'b'): {10}
    }
    
    start_st = 0
    accept_st = {10}

    afn = NFA_Epsilon(states_all, alphabet_in, transitions_nfa, start_st, accept_st)

    print("=== EMULADOR DE AFN-e (Simulación Directa) ===")
    test_string = "aabb"
    print(f"\nEvaluando la cadena: '{test_string}'")
    
    accepted, trace_log = afn.simulate(test_string)
    for line in trace_log:
        print(line)
```

In [ ]:
from typing import Set, Dict, Tuple, List, Any

class NFA_Epsilon:
    """Clase orientada a objetos para modelar y simular un AFN con transiciones epsilon."""
    EPSILON = 'ε'  # Cadena vacía constante

    def __init__(self, states: Set[int], alphabet: Set[str],
                 transitions: Dict[Tuple[int, str], Set[int]],
                 initial_state: int, accepting_states: Set[int]):
        self.S = states
        self.Sigma = alphabet
        self.delta = transitions
        self.s0 = initial_state
        self.F = accepting_states

    def epsilon_closure_state(self, state: int) -> Set[int]:
        """Calcula la epsilon-cerradura para un solo estado usando un algoritmo de pila (DFS)."""
        closure = {state}
        stack = [state]

        while stack:
            current = stack.pop()
            # Buscar transiciones epsilon desde el estado actual
            next_states = self.delta.get((current, self.EPSILON), set())
            for nxt in next_states:
                if nxt not in closure:
                    closure.add(nxt)
                    stack.append(nxt)
        return closure

    def epsilon_closure_set(self, states_set: Set[int]) -> Set[int]:
        """Calcula la epsilon-cerradura para un conjunto de estados: e-closure(T) = U e-closure(s)."""
        total_closure = set()
        for state in states_set:
            total_closure.update(self.epsilon_closure_state(state))
        return total_closure

    def move(self, states_set: Set[int], symbol: str) -> Set[int]:
        """Calcula el conjunto mover(T, a): estados alcanzables con el símbolo 'a'."""
        reachable = set()
        for state in states_set:
            nxt = self.delta.get((state, symbol), set())
            reachable.update(nxt)
        return reachable

    def simulate(self, input_string: str) -> Tuple[bool, List[str]]:
        """Simula la ejecución del AFN (Algoritmo 3.22 del Dragon Book) sobre una cadena."""
        trace = []
        # Paso 1: Estado inicial activo S = e-closure(s0)
        current_active = self.epsilon_closure_state(self.s0)
        trace.append(f"Estado Activo Inicial e-closure({self.s0}): {sorted(list(current_active))}")

        # Bucle principal sobre cada carácter de la entrada
        for i, char in enumerate(input_string):
            if char not in self.Sigma:
                trace.append(f"  [Error] Carácter '{char}' fuera del alfabeto.")
                return False, trace

            # Paso 2a: T = mover(S, c)
            move_set = self.move(current_active, char)
            # Paso 2b: S' = e-closure(T)
            next_active = self.epsilon_closure_set(move_set)

            trace.append(f"  Paso {i+1} ('{char}'): mover -> {sorted(list(move_set))} | e-closure -> {sorted(list(next_active))}")
            current_active = next_active

            if not current_active:
                trace.append("  [Bloqueo] El conjunto de estados activos quedó vacío.")
                return False, trace

        # Criterio de Aceptación: S ∩ F != Ø
        is_accepted = bool(current_active.intersection(self.F))
        status = "CADENA ACEPTADA" if is_accepted else "CADENA RECHAZADA"
        trace.append(f"Estados Activos Finales: {sorted(list(current_active))} | Intersección F: {sorted(list(current_active.intersection(self.F)))}")
        trace.append(f"Resultado: {status}")

        return is_accepted, trace

# --- DEMOSTRACIÓN PRÁCTICA Y EVALUACIÓN ---
if __name__ == "__main__":
    # Construcción del AFN-e para el lenguaje (a|b)*abb
    states_all = {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10}
    alphabet_in = {'a', 'b'}

    # Transiciones del AFN-e (basado en la figura 3.34 de Aho et al.)
    transitions_nfa = {
        (0, NFA_Epsilon.EPSILON): {1, 7},
        (1, NFA_Epsilon.EPSILON): {2, 4},
        (2, 'a'): {3},
        (3, NFA_Epsilon.EPSILON): {6},
        (4, 'b'): {5},
        (5, NFA_Epsilon.EPSILON): {6},
        (6, NFA_Epsilon.EPSILON): {1, 7},
        (7, 'a'): {8},
        (8, 'b'): {9},
        (9, 'b'): {10}
    }

    start_st = 0
    accept_st = {10}

    afn = NFA_Epsilon(states_all, alphabet_in, transitions_nfa, start_st, accept_st)

    print("=== EMULADOR DE AFN-e (Simulación Directa) ===")
    test_string = "aabb"
    print(f"\nEvaluando la cadena: '{test_string}'")

    accepted, trace_log = afn.simulate(test_string)
    for line in trace_log:
        print(line)

=== EMULADOR DE AFN-e (Simulación Directa) ===

Evaluando la cadena: 'aabb'
Estado Activo Inicial e-closure(0): [0, 1, 2, 4, 7]
  Paso 1 ('a'): mover -> [3, 8] | e-closure -> [1, 2, 3, 4, 6, 7, 8]
  Paso 2 ('a'): mover -> [3, 8] | e-closure -> [1, 2, 3, 4, 6, 7, 8]
  Paso 3 ('b'): mover -> [5, 9] | e-closure -> [1, 2, 4, 5, 6, 7, 9]
  Paso 4 ('b'): mover -> [5, 10] | e-closure -> [1, 2, 4, 5, 6, 7, 10]
Estados Activos Finales: [1, 2, 4, 5, 6, 7, 10] | Intersección F: [10]
Resultado: CADENA ACEPTADA


## 4. Sección Práctica 3: ER a AFN-ɛ
En esta sección implementaremos una arquitectura OO para obtener el AFN-ɛ a partir de la ER.

### [CÓDIGO 3.1] ERAFNepsilon.py
```python
# ==============================================================================
# Tema: Algoritmo de Construcción de Thompson
# ==============================================================================

from typing import Set, Dict, Tuple, List

class State:
    """Representa un estado individual de un autómata Thompson."""
    _id_counter = 0

    def __init__(self):
        self.id = State._id_counter
        State._id_counter += 1
        # Diccionario de transiciones: símbolo -> lista de estados destino
        self.transitions: Dict[str, List['State']] = {}

    def add_transition(self, symbol: str, target_state: 'State'):
        if symbol not in self.transitions:
            self.transitions[symbol] = []
        self.transitions[symbol].append(target_state)

    def __repr__(self):
        return f"q{self.id}"

class ThompsonNFA:
    """Encapsula un sub-autómata Thompson con exactamente UN inicio y UN fin."""
    EPSILON = 'ε'

    def __init__(self, start: State, accept: State):
        self.start = start
        self.accept = accept

    @staticmethod
    def build_symbol(symbol: str) -> 'ThompsonNFA':
        """Regla Base: Construcción para un símbolo a in Sigma (o epsilon)."""
        start = State()
        accept = State()
        start.add_transition(symbol, accept)
        return ThompsonNFA(start, accept)

    @staticmethod
    def build_union(nfa1: 'ThompsonNFA', nfa2: 'ThompsonNFA') -> 'ThompsonNFA':
        """Regla Inductiva 1: Unión (nfa1 | nfa2)."""
        start = State()
        accept = State()
        
        # Transiciones epsilon desde nuevo inicio a inicios de nfa1 y nfa2
        start.add_transition(ThompsonNFA.EPSILON, nfa1.start)
        start.add_transition(ThompsonNFA.EPSILON, nfa2.start)
        
        # Transiciones epsilon desde finales de nfa1 y nfa2 a nuevo final
        nfa1.accept.add_transition(ThompsonNFA.EPSILON, accept)
        nfa2.accept.add_transition(ThompsonNFA.EPSILON, accept)
        
        return ThompsonNFA(start, accept)

    @staticmethod
    def build_concat(nfa1: 'ThompsonNFA', nfa2: 'ThompsonNFA') -> 'ThompsonNFA':
        """Regla Inductiva 2: Concatenación (nfa1 . nfa2)."""
        # Conectar final de nfa1 con inicio de nfa2 mediante epsilon
        nfa1.accept.add_transition(ThompsonNFA.EPSILON, nfa2.start)
        return ThompsonNFA(nfa1.start, nfa2.accept)

    @staticmethod
    def build_kleene(nfa: 'ThompsonNFA') -> 'ThompsonNFA':
        """Regla Inductiva 3: Clausura de Kleene (nfa*)."""
        start = State()
        accept = State()
        
        # 1. Transición epsilon de bypass (0 repeticiones)
        start.add_transition(ThompsonNFA.EPSILON, accept)
        # 2. Transición epsilon hacia inicio del sub-autómata
        start.add_transition(ThompsonNFA.EPSILON, nfa.start)
        # 3. Transición epsilon desde final del sub-autómata hacia nuevo final
        nfa.accept.add_transition(ThompsonNFA.EPSILON, accept)
        # 4. Transición epsilon de retroceso (loop de repetición)
        nfa.accept.add_transition(ThompsonNFA.EPSILON, nfa.start)
        
        return ThompsonNFA(start, accept)

    def print_nfa(self):
        """Imprime la lista de adyacencia de todas las transiciones del autómata."""
        visited = set()
        stack = [self.start]
        print(f"\n=== AFN Thompson (Inicio: {self.start}, Aceptación: {self.accept}) ===")
        
        while stack:
            state = stack.pop()
            if state.id in visited:
                continue
            visited.add(state.id)
            
            for symbol, targets in state.transitions.items():
                for target in targets:
                    print(f"  {state} --({symbol})--> {target}")
                    if target.id not in visited:
                        stack.append(target)

# --- DEMOSTRACIÓN PRÁCTICA Y EVALUACIÓN ---
if __name__ == "__main__":
    # Reset contador de estados
    State._id_counter = 0

    # Construir inductivamente el AFN para la expresión: (a | b)*
    nfa_a = ThompsonNFA.build_symbol('a')      # a
    nfa_b = ThompsonNFA.build_symbol('b')      # b
    nfa_ab = ThompsonNFA.build_union(nfa_a, nfa_b)  # (a | b)
    nfa_star = ThompsonNFA.build_kleene(nfa_ab)     # (a | b)*

    nfa_star.print_nfa()
```


In [ ]:
from typing import Set, Dict, Tuple, List

class State:
    """Representa un estado individual de un autómata Thompson."""
    _id_counter = 0

    def __init__(self):
        self.id = State._id_counter
        State._id_counter += 1
        # Diccionario de transiciones: símbolo -> lista de estados destino
        self.transitions: Dict[str, List['State']] = {}

    def add_transition(self, symbol: str, target_state: 'State'):
        if symbol not in self.transitions:
            self.transitions[symbol] = []
        self.transitions[symbol].append(target_state)

    def __repr__(self):
        return f"q{self.id}"

class ThompsonNFA:
    """Encapsula un sub-autómata Thompson con exactamente UN inicio y UN fin."""
    EPSILON = 'ε'

    def __init__(self, start: State, accept: State):
        self.start = start
        self.accept = accept

    @staticmethod
    def build_symbol(symbol: str) -> 'ThompsonNFA':
        """Regla Base: Construcción para un símbolo a in Sigma (o epsilon)."""
        start = State()
        accept = State()
        start.add_transition(symbol, accept)
        return ThompsonNFA(start, accept)

    @staticmethod
    def build_union(nfa1: 'ThompsonNFA', nfa2: 'ThompsonNFA') -> 'ThompsonNFA':
        """Regla Inductiva 1: Unión (nfa1 | nfa2)."""
        start = State()
        accept = State()

        # Transiciones epsilon desde nuevo inicio a inicios de nfa1 y nfa2
        start.add_transition(ThompsonNFA.EPSILON, nfa1.start)
        start.add_transition(ThompsonNFA.EPSILON, nfa2.start)

        # Transiciones epsilon desde finales de nfa1 y nfa2 a nuevo final
        nfa1.accept.add_transition(ThompsonNFA.EPSILON, accept)
        nfa2.accept.add_transition(ThompsonNFA.EPSILON, accept)

        return ThompsonNFA(start, accept)

    @staticmethod
    def build_concat(nfa1: 'ThompsonNFA', nfa2: 'ThompsonNFA') -> 'ThompsonNFA':
        """Regla Inductiva 2: Concatenación (nfa1 . nfa2)."""
        # Conectar final de nfa1 con inicio de nfa2 mediante epsilon
        nfa1.accept.add_transition(ThompsonNFA.EPSILON, nfa2.start)
        return ThompsonNFA(nfa1.start, nfa2.accept)

    @staticmethod
    def build_kleene(nfa: 'ThompsonNFA') -> 'ThompsonNFA':
        """Regla Inductiva 3: Clausura de Kleene (nfa*)."""
        start = State()
        accept = State()

        # 1. Transición epsilon de bypass (0 repeticiones)
        start.add_transition(ThompsonNFA.EPSILON, accept)
        # 2. Transición epsilon hacia inicio del sub-autómata
        start.add_transition(ThompsonNFA.EPSILON, nfa.start)
        # 3. Transición epsilon desde final del sub-autómata hacia nuevo final
        nfa.accept.add_transition(ThompsonNFA.EPSILON, accept)
        # 4. Transición epsilon de retroceso (loop de repetición)
        nfa.accept.add_transition(ThompsonNFA.EPSILON, nfa.start)

        return ThompsonNFA(start, accept)

    def print_nfa(self):
        """Imprime la lista de adyacencia de todas las transiciones del autómata."""
        visited = set()
        stack = [self.start]
        print(f"\n=== AFN Thompson (Inicio: {self.start}, Aceptación: {self.accept}) ===")

        while stack:
            state = stack.pop()
            if state.id in visited:
                continue
            visited.add(state.id)

            for symbol, targets in state.transitions.items():
                for target in targets:
                    print(f"  {state} --({symbol})--> {target}")
                    if target.id not in visited:
                        stack.append(target)

# --- DEMOSTRACIÓN PRÁCTICA Y EVALUACIÓN ---
if __name__ == "__main__":
    # Reset contador de estados
    State._id_counter = 0

    # Construir inductivamente el AFN para la expresión: (a | b)*
    nfa_a = ThompsonNFA.build_symbol('a')      # a
    nfa_b = ThompsonNFA.build_symbol('b')      # b
    nfa_ab = ThompsonNFA.build_union(nfa_a, nfa_b)  # (a | b)
    nfa_star = ThompsonNFA.build_kleene(nfa_ab)     # (a | b)*

    nfa_star.print_nfa()


=== AFN Thompson (Inicio: q6, Aceptación: q7) ===
  q6 --(ε)--> q7
  q6 --(ε)--> q4
  q4 --(ε)--> q0
  q4 --(ε)--> q2
  q2 --(b)--> q3
  q3 --(ε)--> q5
  q5 --(ε)--> q7
  q5 --(ε)--> q4
  q0 --(a)--> q1
  q1 --(ε)--> q5


## 5. Sección Práctica 4: ER a AFN-ɛ
En esta sección implementaremos el Algoritmo de Construcción de Subconjuntos para convertir un AFN en un AFD.

### [CÓDIGO 4.1] AFN-AFD.py
```python
# ==============================================================================
# Tema: Algoritmo de Construcción de Subconjuntos
# ==============================================================================

from typing import Set, Dict, Tuple, List, Optional

class SubsetConstructionConverter:
    """Implementación formal de la Construcción de Subconjuntos (Algoritmo 3.20 Aho et al.)."""
    EPSILON = 'ε'

    def __init__(self, nfa_states: Set[int], alphabet: Set[str],
                 nfa_transitions: Dict[Tuple[int, str], Set[int]],
                 nfa_start_state: int, nfa_accept_states: Set[int]):
        self.nfa_states = nfa_states
        self.alphabet = alphabet
        self.nfa_delta = nfa_transitions
        self.nfa_start = nfa_start_state
        self.nfa_accept = nfa_accept_states

    def e_closure_state(self, state: int) -> Set[int]:
        """Calcula e-closure para un estado individual mediante algoritmo de pila."""
        closure = {state}
        stack = [state]
        while stack:
            curr = stack.pop()
            next_states = self.nfa_delta.get((curr, self.EPSILON), set())
            for nxt in next_states:
                if nxt not in closure:
                    closure.add(nxt)
                    stack.append(nxt)
        return closure

    def e_closure_set(self, states_set: Set[int]) -> Set[int]:
        """Calcula e-closure(T) = U e-closure(s) para s in T."""
        total = set()
        for s in states_set:
            total.update(self.e_closure_state(s))
        return total

    def move(self, states_set: Set[int], symbol: str) -> Set[int]:
        """Calcula mover(T, a): estados alcanzables con 'a' desde T."""
        reachable = set()
        for s in states_set:
            nxt = self.nfa_delta.get((s, symbol), set())
            reachable.update(nxt)
        return reachable

    def convert(self) -> Tuple[List[Set[int]], Dict[Tuple[int, str], int], Set[int]]:
        """Ejecuta el Algoritmo de Construcción de Subconjuntos."""
        # 1. Estado Inicial del AFD: A = e-closure(s0)
        start_closure = self.e_closure_state(self.nfa_start)
        
        d_states: List[Set[int]] = [start_closure]  # Lista de conjuntos Dstates
        d_tran: Dict[Tuple[int, str], int] = {}     # Matriz Dtran[state_idx, symbol]
        unmarked_indices = [0]                      # Cola de estados no marcados

        while unmarked_indices:
            curr_idx = unmarked_indices.pop(0)
            curr_set = d_states[curr_idx]

            for symbol in sorted(list(self.alphabet)):
                # U = e-closure(mover(T, a))
                move_res = self.move(curr_set, symbol)
                u_set = self.e_closure_set(move_res)

                if not u_set:
                    continue  # Transición vacía / no existe

                # Comprobar si U ya existe en Destados
                if u_set in d_states:
                    target_idx = d_states.index(u_set)
                else:
                    d_states.append(u_set)
                    target_idx = len(d_states) - 1
                    unmarked_indices.append(target_idx)

                d_tran[(curr_idx, symbol)] = target_idx

        # 2. Identificar estados finales del AFD (los que contienen al menos un final del AFN)
        dfa_accept_states = set()
        for idx, d_set in enumerate(d_states):
            if bool(d_set.intersection(self.nfa_accept)):
                dfa_accept_states.add(idx)

        return d_states, d_tran, dfa_accept_states

# --- DEMOSTRACIÓN PRÁCTICA Y EVALUACIÓN ---
if __name__ == "__main__":
    # AFN para (a|b)*abb (Ejemplo 3.21 de Aho et al.)
    nfa_S = set(range(11))
    nfa_Sigma = {'a', 'b'}
    nfa_transitions_map = {
        (0, 'ε'): {1, 7},
        (1, 'ε'): {2, 4},
        (2, 'a'): {3},
        (3, 'ε'): {6},
        (4, 'b'): {5},
        (5, 'ε'): {6},
        (6, 'ε'): {1, 7},
        (7, 'a'): {8},
        (8, 'b'): {9},
        (9, 'b'): {10}
    }
    nfa_s0 = 0
    nfa_F = {10}

    converter = SubsetConstructionConverter(nfa_S, nfa_Sigma, nfa_transitions_map, nfa_s0, nfa_F)
    dfa_nodes, dfa_matrix, dfa_accept = converter.convert()

    print("=== RESULTADO DE LA CONSTRUCCIÓN DE SUBCONJUNTOS ===")
    print(f"Total de estados AFD generados: {len(dfa_nodes)}\n")

    state_labels = [chr(65 + i) for i in range(len(dfa_nodes))]  # Nombres A, B, C, D, E...

    for idx, node_set in enumerate(dfa_nodes):
        is_acc = " [ACEPTACIÓN]" if idx in dfa_accept else ""
        print(f"Estado AFD {state_labels[idx]} ({idx}): {sorted(list(node_set))}{is_acc}")

    print("\n=== MATRIZ DE TRANSICIÓN Dtran ===")
    print(f"{'Estado Orig':<12} | {'Símbolo':<8} | {'Estado Destino':<12}")
    print("-" * 40)
    for (src_idx, sym), tgt_idx in dfa_matrix.items():
        print(f"{state_labels[src_idx]} ({src_idx}){'':<6} | {sym:<8} | {state_labels[tgt_idx]} ({tgt_idx})")
```

In [ ]:
from typing import Set, Dict, Tuple, List, Optional

class SubsetConstructionConverter:
    """Implementación formal de la Construcción de Subconjuntos (Algoritmo 3.20 Aho et al.)."""
    EPSILON = 'ε'

    def __init__(self, nfa_states: Set[int], alphabet: Set[str],
                 nfa_transitions: Dict[Tuple[int, str], Set[int]],
                 nfa_start_state: int, nfa_accept_states: Set[int]):
        self.nfa_states = nfa_states
        self.alphabet = alphabet
        self.nfa_delta = nfa_transitions
        self.nfa_start = nfa_start_state
        self.nfa_accept = nfa_accept_states

    def e_closure_state(self, state: int) -> Set[int]:
        """Calcula e-closure para un estado individual mediante algoritmo de pila."""
        closure = {state}
        stack = [state]
        while stack:
            curr = stack.pop()
            next_states = self.nfa_delta.get((curr, self.EPSILON), set())
            for nxt in next_states:
                if nxt not in closure:
                    closure.add(nxt)
                    stack.append(nxt)
        return closure

    def e_closure_set(self, states_set: Set[int]) -> Set[int]:
        """Calcula e-closure(T) = U e-closure(s) para s in T."""
        total = set()
        for s in states_set:
            total.update(self.e_closure_state(s))
        return total

    def move(self, states_set: Set[int], symbol: str) -> Set[int]:
        """Calcula mover(T, a): estados alcanzables con 'a' desde T."""
        reachable = set()
        for s in states_set:
            nxt = self.nfa_delta.get((s, symbol), set())
            reachable.update(nxt)
        return reachable

    def convert(self) -> Tuple[List[Set[int]], Dict[Tuple[int, str], int], Set[int]]:
        """Ejecuta el Algoritmo de Construcción de Subconjuntos."""
        # 1. Estado Inicial del AFD: A = e-closure(s0)
        start_closure = self.e_closure_state(self.nfa_start)

        d_states: List[Set[int]] = [start_closure]  # Lista de conjuntos Dstates
        d_tran: Dict[Tuple[int, str], int] = {}     # Matriz Dtran[state_idx, symbol]
        unmarked_indices = [0]                      # Cola de estados no marcados

        while unmarked_indices:
            curr_idx = unmarked_indices.pop(0)
            curr_set = d_states[curr_idx]

            for symbol in sorted(list(self.alphabet)):
                # U = e-closure(mover(T, a))
                move_res = self.move(curr_set, symbol)
                u_set = self.e_closure_set(move_res)

                if not u_set:
                    continue  # Transición vacía / no existe

                # Comprobar si U ya existe en Destados
                if u_set in d_states:
                    target_idx = d_states.index(u_set)
                else:
                    d_states.append(u_set)
                    target_idx = len(d_states) - 1
                    unmarked_indices.append(target_idx)

                d_tran[(curr_idx, symbol)] = target_idx

        # 2. Identificar estados finales del AFD (los que contienen al menos un final del AFN)
        dfa_accept_states = set()
        for idx, d_set in enumerate(d_states):
            if bool(d_set.intersection(self.nfa_accept)):
                dfa_accept_states.add(idx)

        return d_states, d_tran, dfa_accept_states

# --- DEMOSTRACIÓN PRÁCTICA Y EVALUACIÓN ---
if __name__ == "__main__":
    # AFN para (a|b)*abb (Ejemplo 3.21 de Aho et al.)
    nfa_S = set(range(11))
    nfa_Sigma = {'a', 'b'}
    nfa_transitions_map = {
        (0, 'ε'): {1, 7},
        (1, 'ε'): {2, 4},
        (2, 'a'): {3},
        (3, 'ε'): {6},
        (4, 'b'): {5},
        (5, 'ε'): {6},
        (6, 'ε'): {1, 7},
        (7, 'a'): {8},
        (8, 'b'): {9},
        (9, 'b'): {10}
    }
    nfa_s0 = 0
    nfa_F = {10}

    converter = SubsetConstructionConverter(nfa_S, nfa_Sigma, nfa_transitions_map, nfa_s0, nfa_F)
    dfa_nodes, dfa_matrix, dfa_accept = converter.convert()

    print("=== RESULTADO DE LA CONSTRUCCIÓN DE SUBCONJUNTOS ===")
    print(f"Total de estados AFD generados: {len(dfa_nodes)}\n")

    state_labels = [chr(65 + i) for i in range(len(dfa_nodes))]  # Nombres A, B, C, D, E...

    for idx, node_set in enumerate(dfa_nodes):
        is_acc = " [ACEPTACIÓN]" if idx in dfa_accept else ""
        print(f"Estado AFD {state_labels[idx]} ({idx}): {sorted(list(node_set))}{is_acc}")

    print("\n=== MATRIZ DE TRANSICIÓN Dtran ===")
    print(f"{'Estado Orig':<12} | {'Símbolo':<8} | {'Estado Destino':<12}")
    print("-" * 40)
    for (src_idx, sym), tgt_idx in dfa_matrix.items():
        print(f"{state_labels[src_idx]} ({src_idx}){'':<6} | {sym:<8} | {state_labels[tgt_idx]} ({tgt_idx})")

=== RESULTADO DE LA CONSTRUCCIÓN DE SUBCONJUNTOS ===
Total de estados AFD generados: 5

Estado AFD A (0): [0, 1, 2, 4, 7]
Estado AFD B (1): [1, 2, 3, 4, 6, 7, 8]
Estado AFD C (2): [1, 2, 4, 5, 6, 7]
Estado AFD D (3): [1, 2, 4, 5, 6, 7, 9]
Estado AFD E (4): [1, 2, 4, 5, 6, 7, 10] [ACEPTACIÓN]

=== MATRIZ DE TRANSICIÓN Dtran ===
Estado Orig  | Símbolo  | Estado Destino
----------------------------------------
A (0)       | a        | B (1)
A (0)       | b        | C (2)
B (1)       | a        | B (1)
B (1)       | b        | D (3)
C (2)       | a        | B (1)
C (2)       | b        | C (2)
D (3)       | a        | B (1)
D (3)       | b        | E (4)
E (4)       | a        | B (1)
E (4)       | b        | C (2)


## Desafío 1: Exploración y Modificación de un AFD (Nivel: Básico)
Script de referencia: AFDeterminista.py

**Objetivo:** Comprender la estructura de una 5-tupla, el manejo de diccionarios como funciones de transición y la representación tabular.

**Consigna para el estudiante:**

>Ejecución y análisis: Capture y ejecute el programa original. Observe detenidamente cómo se imprime la matriz de transición (`Dtran`) y cómo se evalúan las cadenas `"LLD"`, `"L1D"`, etc.

>Modificación del alfabeto y el lenguaje: Modifique el autómata para que, además de reconocer identificadores simples (`L(L|D)*`), acepte también números enteros positivos (por ejemplo, una secuencia de uno o más dígitos: `D+`).

>Ampliación de pruebas: Agregue al arreglo `test_inputs` al menos tres cadenas adicionales (una válida que comience con dígitos, otra mixta y una inválida) y documente paso a paso mediante comentarios en su código por qué el AFD rechaza o acepta cada una.

In [ ]:
# Desafío 1: Exploración y Modificación de un AFD
from typing import Set, Dict, Tuple, List


class AutomataFinitoDeterminista:
    """Clase orientada a objetos para modelar y simular un AFD (5-tupla)."""

    def __init__(self, states: Set[int], alphabet: Set[str],
                 transitions: Dict[Tuple[int, str], int],
                 initial_state: int, accepting_states: Set[int],
                 token_names: Dict[int, str] = None):
        self.S = states                 # estados
        self.Sigma = alphabet           # alfabeto
        self.delta = transitions        # función de transición
        self.s0 = initial_state         # estado inicial
        self.F = accepting_states       # estados de aceptación

        # Le agregué esto para que al aceptar una cadena diga si fue
        # identificador o entero, así se entiende mejor la salida
        self.token_names = token_names or {}

        # Estado de Error implícito
        self.ERROR_STATE = -1

    def transition(self, state: int, symbol: str) -> int:
        """Consulta la función/matriz de transición delta(s, c)."""
        return self.delta.get((state, symbol), self.ERROR_STATE)

    def evaluate_string(self, input_string: str) -> Tuple[bool, List[str]]:
        """Simula la ejecución del AFD carácter a carácter trazando los estados."""
        current_state = self.s0
        trace = [f"Estado Inicial: q{current_state}"]

        for i, char in enumerate(input_string):
            if char not in self.Sigma:
                trace.append(f"  [Error] Carácter '{char}' fuera del alfabeto Sigma.")
                return False, trace

            next_state = self.transition(current_state, char)
            if next_state == self.ERROR_STATE:
                trace.append(f"  [Bloqueo] Sin transición desde q{current_state} con '{char}'.")
                return False, trace

            trace.append(f"  Paso {i+1}: Símbolo '{char}' | Transición q{current_state} -> q{next_state}")
            current_state = next_state

        is_accepted = current_state in self.F
        status = "CADENA ACEPTADA" if is_accepted else "CADENA RECHAZADA"
        if is_accepted and current_state in self.token_names:
            status += f" ({self.token_names[current_state]})"
        trace.append(f"Estado Final: q{current_state} | Resultado: {status}")
        return is_accepted, trace

    def print_transition_matrix(self):
        """Imprime la representación tabular/matriz de transición del AFD."""
        sorted_symbols = sorted(list(self.Sigma))
        sorted_states = sorted(list(self.S))

        print("\n=== MATRIZ DE TRANSICIÓN Dtran[s, c] ===")
        header = f"{'Estado':<8} | " + " | ".join([f"{sym:^5}" for sym in sorted_symbols])
        print(header)
        print("-" * len(header))

        for state in sorted_states:
            prefix = "-> " if state == self.s0 else (" * " if state in self.F else "   ")
            row_str = f"{prefix}q{state:<4} | "
            row_cells = []
            for sym in sorted_symbols:
                dest = self.transition(state, sym)
                cell = f"q{dest}" if dest != self.ERROR_STATE else "--"
                row_cells.append(f"{cell:^5}")
            row_str += " | ".join(row_cells)
            print(row_str)
        print("(-> estado inicial, * estado de aceptación, -- sin transición)")


# --- DEMOSTRACIÓN PRÁCTICA Y EVALUACIÓN ---
if __name__ == "__main__":
    # Para que también acepte números enteros agregué el estado q2.
    # Si la cadena empieza con letra se va a q1 (identificadores) y si
    # empieza con dígito se va a q2 (enteros). En q2 solo puede seguir
    # leyendo dígitos, si llega una letra se rechaza.
    # El alfabeto sigue siendo {L, D}, nada más cambió lo que acepta.
    states_set = {0, 1, 2}
    alphabet_set = {'L', 'D'}

    transitions_map = {
        (0, 'L'): 1,   # empieza con letra, va a identificadores
        (1, 'L'): 1,   # un identificador puede seguir con letras
        (1, 'D'): 1,   # o con dígitos
        (0, 'D'): 2,   # nueva: empieza con dígito, va a enteros
        (2, 'D'): 2,   # nueva: un entero solo puede seguir con dígitos
        # No puse (2, 'L') a propósito, porque algo como "12a" no es
        # ni entero ni identificador y tiene que rechazarse
    }

    initial = 0
    accepting = {1, 2}  # q2 también acepta, porque ahí terminan los enteros

    tokens = {1: "IDENTIFICADOR", 2: "ENTERO"}

    afd = AutomataFinitoDeterminista(states_set, alphabet_set, transitions_map,
                                     initial, accepting, tokens)

    # Mostrar la matriz de transición
    afd.print_transition_matrix()

    # Batería de pruebas

    test_inputs = [
        # Las del programa original
        "LLD",    # queda "LLL": entra a q1 y ahí se queda, se acepta
        "L1D",    # queda "LDL": empieza con letra y sigue en q1, se acepta
        "DLL",    # queda "LLL": la D es letra, no dígito, así que se acepta igual
        "L",      # queda "L": de q0 pasa a q1 y termina, se acepta

        # Las que agregué
        "2026",   # Válida que empieza con dígitos. Queda "DDDD": de q0 pasa a
                  # q2 con la primera D y se queda ahí con las demás. Termina
                  # en q2, que es de aceptación, así que es un entero válido.
                  # Con el AFD original esta se rechazaba porque q0 no tenía
                  # a dónde ir con una D.

        "x9y8",   # Mixta. Queda "LDLD": como empieza con letra se va a q1, y
                  # en q1 da igual si viene letra o dígito, se queda ahí.
                  # Termina en q1, entonces se acepta como identificador.

        "7up",    # Inválida. Queda "DLL": con el 7 pasa a q2, pero luego
                  # llega una letra y q2 no tiene transición con L, así que el
                  # autómata se atora y la rechaza. No es entero porque tiene
                  # letras, y tampoco identificador porque empieza con número.

        "var_1",  # Inválida. Las primeras letras van bien en q1, pero el
                  # guion bajo no es letra ni dígito, entonces no está en el
                  # alfabeto y se rechaza ahí mismo.
    ]

    print("\n=== EVALUACIÓN DE CADENAS DE PRUEBA ===")
    for test in test_inputs:
        # Convertir caracteres para simplificar la prueba (letras -> 'L', dígitos -> 'D')
        mapped_input = "".join(['L' if c.isalpha() else ('D' if c.isdigit() else c) for c in test])
        print(f"\nEvaluando Entrada Original: '{test}' -> Abstracta: '{mapped_input}'")
        accepted, execution_trace = afd.evaluate_string(mapped_input)
        for line in execution_trace:
            print(line)


=== MATRIZ DE TRANSICIÓN Dtran[s, c] ===
Estado   |   D   |   L  
------------------------
-> q0    |  q2   |  q1  
 * q1    |  q1   |  q1  
 * q2    |  q2   |  --  
(-> estado inicial, * estado de aceptación, -- sin transición)

=== EVALUACIÓN DE CADENAS DE PRUEBA ===

Evaluando Entrada Original: 'LLD' -> Abstracta: 'LLL'
Estado Inicial: q0
  Paso 1: Símbolo 'L' | Transición q0 -> q1
  Paso 2: Símbolo 'L' | Transición q1 -> q1
  Paso 3: Símbolo 'L' | Transición q1 -> q1
Estado Final: q1 | Resultado: CADENA ACEPTADA (IDENTIFICADOR)

Evaluando Entrada Original: 'L1D' -> Abstracta: 'LDL'
Estado Inicial: q0
  Paso 1: Símbolo 'L' | Transición q0 -> q1
  Paso 2: Símbolo 'D' | Transición q1 -> q1
  Paso 3: Símbolo 'L' | Transición q1 -> q1
Estado Final: q1 | Resultado: CADENA ACEPTADA (IDENTIFICADOR)

Evaluando Entrada Original: 'DLL' -> Abstracta: 'LLL'
Estado Inicial: q0
  Paso 1: Símbolo 'L' | Transición q0 -> q1
  Paso 2: Símbolo 'L' | Transición q1 -> q1
  Paso 3: Símbolo 'L' | Transic

##Desafío 2: Rastreo de Pila y Clausuras Epsilon en AFN (Nivel: Básico-Intermedio)  

Script de referencia: epsilonClosure.py  

**Objetivo:** Analizar el algoritmo basado en pilas (DFS) para calcular la épsilon-clausura y entender la simulación de caminos múltiples no deterministas.  

**Consigna para el estudiante:**  

>Trazado manual vs. computacional: Siga el rastro impreso por el emulador para la cadena `"aabb"`. Identifique visualmente en qué momento exacto el conjunto de estados activos se bifurca gracias a las transiciones $\varepsilon$.  

>Instrumentación del código: Modifique el método `epsilon_closure_state` para que imprima en consola cada vez que un estado entra y sale de la pila (`stack`). Esto le permitirá visualizar el recorrido tipo DFS (Depth-First Search).  

>Prueba de estrés de diseño: Cambie la cadena de prueba a `"aa"` y a `"abbb"`. Analice la traza de ejecución e identifique en qué paso exacto ocurre un `[Bloqueo]` o por qué se rechaza la cadena según el criterio de intersección con el conjunto de aceptación `F`.

In [11]:
# Desafío 2: Rastreo de Pila y Clausuras Epsilon en AFN

from typing import Set, Dict, Tuple, List, Any

class NFA_Epsilon:
    """Clase orientada a objetos para modelar y simular un AFN con transiciones epsilon."""
    EPSILON = 'ε'  # Cadena vacía constante

    def __init__(self, states: Set[int], alphabet: Set[str],
                 transitions: Dict[Tuple[int, str], Set[int]],
                 initial_state: int, accepting_states: Set[int],
                 verbose: bool = True):
        self.S = states
        self.Sigma = alphabet
        self.delta = transitions
        self.s0 = initial_state
        self.F = accepting_states
        # Con esto prendo o apago los mensajes de la pila, para que no
        # salgan si nada más quiero ver el resultado
        self.verbose = verbose

    def epsilon_closure_state(self, state: int) -> Set[int]:
        """Calcula la epsilon-cerradura para un solo estado usando un algoritmo de pila (DFS)."""
        closure = {state}
        stack = [state]
        if self.verbose:
            print(f"    e-closure(q{state}):")
            print(f"      push q{state}  -> pila: {self._pila(stack)}")

        while stack:
            current = stack.pop()
            if self.verbose:
                print(f"      pop  q{current}  -> pila: {self._pila(stack)}")
            # Buscar transiciones epsilon desde el estado actual
            next_states = self.delta.get((current, self.EPSILON), set())
            # Los recorro ordenados para que la traza salga siempre igual
            for nxt in sorted(next_states):
                if nxt not in closure:
                    closure.add(nxt)
                    stack.append(nxt)
                    if self.verbose:
                        print(f"      push q{nxt}  -> pila: {self._pila(stack)}   (q{current} --ε--> q{nxt})")
        if self.verbose:
            print(f"      pila vacía, resultado: {sorted(closure)}")
        return closure

    def _pila(self, stack: List[int]) -> str:
        """Muestra la pila como texto, el tope es el último de la derecha."""
        return "[" + ", ".join(f"q{s}" for s in stack) + "]"

    def epsilon_closure_set(self, states_set: Set[int]) -> Set[int]:
        """Calcula la epsilon-cerradura para un conjunto de estados: e-closure(T) = U e-closure(s)."""
        total_closure = set()
        for state in sorted(states_set):
            total_closure.update(self.epsilon_closure_state(state))
        return total_closure

    def move(self, states_set: Set[int], symbol: str) -> Set[int]:
        """Calcula el conjunto mover(T, a): estados alcanzables con el símbolo 'a'."""
        reachable = set()
        for state in states_set:
            nxt = self.delta.get((state, symbol), set())
            reachable.update(nxt)
        return reachable

    def simulate(self, input_string: str) -> Tuple[bool, List[str]]:
        """Simula la ejecución del AFN (Algoritmo 3.22 del Dragon Book) sobre una cadena."""
        trace = []

        # Antes la traza se imprimía hasta el final, y los mensajes de la
        # pila salían todos amontonados arriba. Así cada línea se imprime
        # en el momento y queda en orden con los push/pop.
        def registrar(msg: str):
            trace.append(msg)
            print(msg)

        # Paso 1: Estado inicial activo S = e-closure(s0)
        current_active = self.epsilon_closure_state(self.s0)
        registrar(f"Estado Activo Inicial e-closure({self.s0}): {sorted(list(current_active))}")

        # Bucle principal sobre cada carácter de la entrada
        for i, char in enumerate(input_string):
            if char not in self.Sigma:
                registrar(f"  [Error] Carácter '{char}' fuera del alfabeto.")
                return False, trace

            # Paso 2a: T = mover(S, c)
            move_set = self.move(current_active, char)
            if self.verbose:
                print(f"\n  -- Paso {i+1} ('{char}'): mover da {sorted(move_set)}, calculando su e-closure --")
            # Paso 2b: S' = e-closure(T)
            next_active = self.epsilon_closure_set(move_set)

            registrar(f"  Paso {i+1} ('{char}'): mover -> {sorted(list(move_set))} | e-closure -> {sorted(list(next_active))}")
            current_active = next_active

            if not current_active:
                registrar("  [Bloqueo] El conjunto de estados activos quedó vacío.")
                return False, trace

        # Criterio de Aceptación: S ∩ F != Ø
        is_accepted = bool(current_active.intersection(self.F))
        status = "CADENA ACEPTADA" if is_accepted else "CADENA RECHAZADA"
        registrar(f"Estados Activos Finales: {sorted(list(current_active))} | Intersección F: {sorted(list(current_active.intersection(self.F)))}")
        registrar(f"Resultado: {status}")

        return is_accepted, trace

# --- DEMOSTRACIÓN PRÁCTICA Y EVALUACIÓN ---
if __name__ == "__main__":
    # Construcción del AFN-e para el lenguaje (a|b)*abb
    states_all = {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10}
    alphabet_in = {'a', 'b'}

    # Transiciones del AFN-e (basado en la figura 3.34 de Aho et al.)
    transitions_nfa = {
        (0, NFA_Epsilon.EPSILON): {1, 7},
        (1, NFA_Epsilon.EPSILON): {2, 4},
        (2, 'a'): {3},
        (3, NFA_Epsilon.EPSILON): {6},
        (4, 'b'): {5},
        (5, NFA_Epsilon.EPSILON): {6},
        (6, NFA_Epsilon.EPSILON): {1, 7},
        (7, 'a'): {8},
        (8, 'b'): {9},
        (9, 'b'): {10}
    }

    start_st = 0
    accept_st = {10}

    afn = NFA_Epsilon(states_all, alphabet_in, transitions_nfa, start_st, accept_st)

    print("=== EMULADOR DE AFN-e (Simulación Directa) ===")

    # "aabb" es la cadena original, "aa" y "abbb" son las de la prueba de estrés.
    # En este AFN nunca va a salir [Bloqueo] con a's y b's, porque el ciclo
    # (a|b)* siempre deja estados activos. Por eso las dos se rechazan por
    # no llegar a q10 (S ∩ F vacía), no porque se queden sin estados.

    test_strings = [
        "aabb",   # Se acepta. Las ε hacen que desde el inicio haya varios caminos
                  # vivos al mismo tiempo (q0 se abre a q1 y q7, y q1 a q2 y q4).
                  # Al final queda q10 entre los activos, entonces S ∩ F = {10}.

        "aa",     # Se rechaza, pero no por bloqueo. Después de cada 'a' los
                  # activos son [1, 2, 3, 4, 6, 7, 8], nunca se vacían porque el
                  # ciclo (a|b)* siempre deja a q1, q2, q4 y q7 vivos. El problema
                  # es que q10 no aparece, así que S ∩ F queda vacía.

        "abbb",   # Se rechaza también por la intersección. Con "abb" sí se llega
                  # a q10 (en el paso 3), pero la última 'b' lo saca: q10 no tiene
                  # transiciones, entonces mover lo pierde y solo sobreviven los
                  # caminos del ciclo. Termina en [1, 2, 4, 5, 6, 7] sin q10.
    ]

    for test_string in test_strings:
        print("\n" + "=" * 70)
        print(f"Evaluando la cadena: '{test_string}'")
        print("=" * 70)
        afn.simulate(test_string)

=== EMULADOR DE AFN-e (Simulación Directa) ===

Evaluando la cadena: 'aabb'
    e-closure(q0):
      push q0  -> pila: [q0]
      pop  q0  -> pila: []
      push q1  -> pila: [q1]   (q0 --ε--> q1)
      push q7  -> pila: [q1, q7]   (q0 --ε--> q7)
      pop  q7  -> pila: [q1]
      pop  q1  -> pila: []
      push q2  -> pila: [q2]   (q1 --ε--> q2)
      push q4  -> pila: [q2, q4]   (q1 --ε--> q4)
      pop  q4  -> pila: [q2]
      pop  q2  -> pila: []
      pila vacía, resultado: [0, 1, 2, 4, 7]
Estado Activo Inicial e-closure(0): [0, 1, 2, 4, 7]

  -- Paso 1 ('a'): mover da [3, 8], calculando su e-closure --
    e-closure(q3):
      push q3  -> pila: [q3]
      pop  q3  -> pila: []
      push q6  -> pila: [q6]   (q3 --ε--> q6)
      pop  q6  -> pila: []
      push q1  -> pila: [q1]   (q6 --ε--> q1)
      push q7  -> pila: [q1, q7]   (q6 --ε--> q7)
      pop  q7  -> pila: [q1]
      pop  q1  -> pila: []
      push q2  -> pila: [q2]   (q1 --ε--> q2)
      push q4  -> pila: [q2, q4]   (q1

### Desafío 3: Construcción de Subconjuntos y Equivalencia de Autómatas (Nivel: Intermedio)  

Script de referencia: afnafd.py y AFDeterminista.py  

**Objetivo:** Conectar la teoría del paso de AFN a AFD mediante el algoritmo de construcción de subconjuntos, analizando la explosión o optimización de estados.  

**Consigna para el estudiante:**  

>Análisis de salida: Ejecute `afnafd.py` y examine la correspondencia entre los subconjuntos numéricos generados (p. ej., `{0, 1, 2, 4, 7}`) y las etiquetas alfabéticas asignadas (`A`, `B`, `C`, etc.).  

>Validación cruzada: Tome la matriz de transición resultante (`d_tran`) generada por el script de subconjuntos para el lenguaje $(a\vert{}b)^*abb$ y conviértala manualmente a una tabla tabular similar a la que imprime `AFDeterminista.py`.  

>Pregunta de reflexión técnica: En su reporte de práctica, responda: ¿Por qué el algoritmo utiliza una cola de índices no marcados (`unmarked_indices`) y qué pasaría con la ejecución si omitimos la verificación `if u_set in d_states`?

¿Por qué se usa la cola unmarked_indices?.

R: Porque cada vez que aparece un subconjunto nuevo todavía no se sabe a dónde va con cada símbolo. La cola guarda los estados que faltan por revisar (“no marcados”). El algoritmo saca uno, calcula sus transiciones con a y b, y si salen subconjuntos nuevos los mete a la cola. Cuando la cola se vacía significa que ya se revisaron todos los estados y la tabla está completa. Como se saca con pop(0), se recorren en orden de aparición (como un BFS), y por eso las letras A, B, C... quedan en ese orden.

¿Qué pasa si se quita if u_set in d_states?.

R: Cada conjunto que se calcula se agregaría como estado nuevo aunque ya existiera, y se volvería a meter a la cola. En este autómata todos los estados tienen transición con a y con b, así que cada estado que se revisa genera dos más y la cola nunca se vacía: el programa se queda en un ciclo infinito. Lo probé limitándolo a 2000 iteraciones y ya llevaba 4001 estados creados, con 2001 pendientes en la cola, cuando en realidad solo existen 5 distintos. Esa verificación es lo que hace que el algoritmo termine y no cree estados repetidos.

In [13]:
# Desafío 3: Validación cruzada

from typing import Set, Dict, Tuple, List, Optional

class AutomataFinitoDeterminista:
    """Clase orientada a objetos para modelar y simular un AFD (5-tupla)."""

    def __init__(self, states: Set[int], alphabet: Set[str],
                 transitions: Dict[Tuple[int, str], int],
                 initial_state: int, accepting_states: Set[int]):
        self.S = states
        self.Sigma = alphabet
        self.delta = transitions  # Corregido: atributo asignado correctamente
        self.s0 = initial_state
        self.F = accepting_states

        # Estado de Error implícito
        self.ERROR_STATE = -1

    def transition(self, state: int, symbol: str) -> int:
        """Consulta la función/matriz de transición delta(s, c)."""
        return self.delta.get((state, symbol), self.ERROR_STATE) # Corregido: llamado al atributo delta

    def evaluate_string(self, input_string: str) -> Tuple[bool, List[str]]:
        """Simula la ejecución del AFD carácter a carácter trazando los estados."""
        current_state = self.s0
        trace = [f"Estado Inicial: q{current_state}"]

        for i, char in enumerate(input_string):
            if char not in self.Sigma:
                trace.append(f"  [Error] Carácter '{char}' fuera del alfabeto Sigma.")
                return False, trace

            next_state = self.transition(current_state, char)
            if next_state == self.ERROR_STATE:
                trace.append(f"  [Bloqueo] Sin transición desde q{current_state} con '{char}'.")
                return False, trace

            trace.append(f"  Paso {i+1}: Símbolo '{char}' | Transición q{current_state} -> q{next_state}")
            current_state = next_state

        is_accepted = current_state in self.F
        status = "CADENA ACEPTADA" if is_accepted else "CADENA RECHAZADA"
        trace.append(f"Estado Final: q{current_state} | Resultado: {status}")
        return is_accepted, trace

    def print_transition_matrix(self):
        """Imprime la representación tabular/matriz de transición del AFD."""
        sorted_symbols = sorted(list(self.Sigma))
        sorted_states = sorted(list(self.S))

        print("\n=== MATRIZ DE TRANSICIÓN Dtran[s, c] ===")
        header = f"{'Estado':<8} | " + " | ".join([f"{sym:^5}" for sym in sorted_symbols])
        print(header)
        print("-" * len(header))

        for state in sorted_states:
            prefix = "-> " if state == self.s0 else (" * " if state in self.F else "   ")
            row_str = f"{prefix}q{state:<4} | "
            row_cells = []
            for sym in sorted_symbols:
                dest = self.transition(state, sym)
                cell = f"q{dest}" if dest != self.ERROR_STATE else "--"
                row_cells.append(f"{cell:^5}")
            row_str += " | ".join(row_cells)
            print(row_str)

# Así quedaron los estados después de correr afnafd.py. Como la clase de
# AFDeterminista usa números, A es q0, B es q1, y así:
#   A (q0) = {0, 1, 2, 4, 7}            -> estado inicial
#   B (q1) = {1, 2, 3, 4, 6, 7, 8}
#   C (q2) = {1, 2, 4, 5, 6, 7}
#   D (q3) = {1, 2, 4, 5, 6, 7, 9}
#   E (q4) = {1, 2, 4, 5, 6, 7, 10}     -> de aceptación, es el único que tiene al 10

estados = {0, 1, 2, 3, 4}
alfabeto = {'a', 'b'}

# Copié a mano cada renglón de la matriz Dtran que imprimió afnafd.py
transiciones = {
    (0, 'a'): 1,   # A --a--> B
    (0, 'b'): 2,   # A --b--> C
    (1, 'a'): 1,   # B --a--> B
    (1, 'b'): 3,   # B --b--> D
    (2, 'a'): 1,   # C --a--> B
    (2, 'b'): 2,   # C --b--> C
    (3, 'a'): 1,   # D --a--> B
    (3, 'b'): 4,   # D --b--> E
    (4, 'a'): 1,   # E --a--> B
    (4, 'b'): 2,   # E --b--> C
}

inicial = 0
aceptacion = {4}

afd = AutomataFinitoDeterminista(estados, alfabeto, transiciones, inicial, aceptacion)
afd.print_transition_matrix()

# Uso las mismas cadenas del desafío 2 para comparar. Si el AFD da el mismo
# resultado que el AFN, entonces los dos son equivalentes.
#   aabb -> el AFN la aceptó
#   aa   -> el AFN la rechazó
#   abbb -> el AFN la rechazó
# y agregué abb y babb, que también deberían aceptarse
cadenas = ["aabb", "aa", "abbb", "abb", "babb"]

print("\n=== EVALUACIÓN CON EL AFD OBTENIDO ===")
for cadena in cadenas:
    print(f"\nCadena: '{cadena}'")
    aceptada, traza = afd.evaluate_string(cadena)
    for linea in traza:
        print(linea)


=== MATRIZ DE TRANSICIÓN Dtran[s, c] ===
Estado   |   a   |   b  
------------------------
-> q0    |  q1   |  q2  
   q1    |  q1   |  q3  
   q2    |  q1   |  q2  
   q3    |  q1   |  q4  
 * q4    |  q1   |  q2  

=== EVALUACIÓN CON EL AFD OBTENIDO ===

Cadena: 'aabb'
Estado Inicial: q0
  Paso 1: Símbolo 'a' | Transición q0 -> q1
  Paso 2: Símbolo 'a' | Transición q1 -> q1
  Paso 3: Símbolo 'b' | Transición q1 -> q3
  Paso 4: Símbolo 'b' | Transición q3 -> q4
Estado Final: q4 | Resultado: CADENA ACEPTADA

Cadena: 'aa'
Estado Inicial: q0
  Paso 1: Símbolo 'a' | Transición q0 -> q1
  Paso 2: Símbolo 'a' | Transición q1 -> q1
Estado Final: q1 | Resultado: CADENA RECHAZADA

Cadena: 'abbb'
Estado Inicial: q0
  Paso 1: Símbolo 'a' | Transición q0 -> q1
  Paso 2: Símbolo 'b' | Transición q1 -> q3
  Paso 3: Símbolo 'b' | Transición q3 -> q4
  Paso 4: Símbolo 'b' | Transición q4 -> q2
Estado Final: q2 | Resultado: CADENA RECHAZADA

Cadena: 'abb'
Estado Inicial: q0
  Paso 1: Símbolo 'a' | Tr

### Desafío 4: Extensión de la Construcción de Thompson (Nivel: Intermedio)  

Script de referencia: erafnEpsilon.py  

**Objetivo:** Entender la programación orientada a objetos aplicada a estructuras de grafos y la aplicación práctica de reglas inductivas en compiladores.  

**Consigna para el estudiante:**  

>Estudio de arquitectura POO: Analice cómo la clase `State` gestiona las referencias a objetos vecinos mediante listas de adyacencia y cómo la clase `ThompsonNFA` encapsula los puntos de entrada (`start`) y salida (`accept`).  

>Desafío de implementación (Operador Opcional `?`): Añada un nuevo método estático a la clase `ThompsonNFA` llamado `build_optional(nfa: 'ThompsonNFA') -> 'ThompsonNFA'` que implemente la regla inductiva para la cerradura opcional ($r?$, es decir, cero o una repetición). Pista: requiere conectar el nuevo estado inicial con el estado inicial del sub-autómata y directamente con el nuevo estado final mediante épsilon, además de conectar el final anterior al nuevo final.  

>Prueba de integración: Utilice su nuevo método para construir el AFN correspondiente a la expresión regular opcional `a?` y ejecute la impresión de la lista de adyacencia para verificar sus conexiones.

In [14]:
# Desafío 4: Extensión con el operador opcional (?)

from typing import Set, Dict, Tuple, List

class State:
    """Representa un estado individual de un autómata Thompson."""
    _id_counter = 0

    def __init__(self):
        self.id = State._id_counter
        State._id_counter += 1
        # Diccionario de transiciones: símbolo -> lista de estados destino
        self.transitions: Dict[str, List['State']] = {}

    def add_transition(self, symbol: str, target_state: 'State'):
        if symbol not in self.transitions:
            self.transitions[symbol] = []
        self.transitions[symbol].append(target_state)

    def __repr__(self):
        return f"q{self.id}"

class ThompsonNFA:
    """Encapsula un sub-autómata Thompson con exactamente UN inicio y UN fin."""
    EPSILON = 'ε'

    def __init__(self, start: State, accept: State):
        self.start = start
        self.accept = accept

    @staticmethod
    def build_symbol(symbol: str) -> 'ThompsonNFA':
        """Regla Base: Construcción para un símbolo a in Sigma (o epsilon)."""
        start = State()
        accept = State()
        start.add_transition(symbol, accept)
        return ThompsonNFA(start, accept)

    @staticmethod
    def build_union(nfa1: 'ThompsonNFA', nfa2: 'ThompsonNFA') -> 'ThompsonNFA':
        """Regla Inductiva 1: Unión (nfa1 | nfa2)."""
        start = State()
        accept = State()

        # Transiciones epsilon desde nuevo inicio a inicios de nfa1 y nfa2
        start.add_transition(ThompsonNFA.EPSILON, nfa1.start)
        start.add_transition(ThompsonNFA.EPSILON, nfa2.start)

        # Transiciones epsilon desde finales de nfa1 y nfa2 a nuevo final
        nfa1.accept.add_transition(ThompsonNFA.EPSILON, accept)
        nfa2.accept.add_transition(ThompsonNFA.EPSILON, accept)

        return ThompsonNFA(start, accept)

    @staticmethod
    def build_concat(nfa1: 'ThompsonNFA', nfa2: 'ThompsonNFA') -> 'ThompsonNFA':
        """Regla Inductiva 2: Concatenación (nfa1 . nfa2)."""
        # Conectar final de nfa1 con inicio de nfa2 mediante epsilon
        nfa1.accept.add_transition(ThompsonNFA.EPSILON, nfa2.start)
        return ThompsonNFA(nfa1.start, nfa2.accept)

    @staticmethod
    def build_kleene(nfa: 'ThompsonNFA') -> 'ThompsonNFA':
        """Regla Inductiva 3: Clausura de Kleene (nfa*)."""
        start = State()
        accept = State()

        # 1. Transición epsilon de bypass (0 repeticiones)
        start.add_transition(ThompsonNFA.EPSILON, accept)
        # 2. Transición epsilon hacia inicio del sub-autómata
        start.add_transition(ThompsonNFA.EPSILON, nfa.start)
        # 3. Transición epsilon desde final del sub-autómata hacia nuevo final
        nfa.accept.add_transition(ThompsonNFA.EPSILON, accept)
        # 4. Transición epsilon de retroceso (loop de repetición)
        nfa.accept.add_transition(ThompsonNFA.EPSILON, nfa.start)

        return ThompsonNFA(start, accept)

    @staticmethod
    def build_optional(nfa: 'ThompsonNFA') -> 'ThompsonNFA':
        """Regla Inductiva 4: Opcional (nfa?), o sea cero o una vez."""
        start = State()
        accept = State()

        # Del inicio nuevo entro al sub-autómata, este es el caso de una vez
        start.add_transition(ThompsonNFA.EPSILON, nfa.start)
        # Del inicio nuevo me salto directo al final, este es el caso de cero veces
        start.add_transition(ThompsonNFA.EPSILON, accept)
        # Cuando el sub-autómata termina, se conecta al final nuevo
        nfa.accept.add_transition(ThompsonNFA.EPSILON, accept)

        # Es casi igual que build_kleene, pero sin la transición que regresa
        # del final al inicio del sub-autómata. Sin esa vuelta no se puede
        # repetir, así que solo se puede pasar una vez o ninguna.
        return ThompsonNFA(start, accept)

    def print_nfa(self):
        """Imprime la lista de adyacencia de todas las transiciones del autómata."""
        visited = set()
        stack = [self.start]
        print(f"\n=== AFN Thompson (Inicio: {self.start}, Aceptación: {self.accept}) ===")

        while stack:
            state = stack.pop()
            if state.id in visited:
                continue
            visited.add(state.id)

            for symbol, targets in state.transitions.items():
                for target in targets:
                    print(f"  {state} --({symbol})--> {target}")
                    if target.id not in visited:
                        stack.append(target)

# --- DEMOSTRACIÓN PRÁCTICA Y EVALUACIÓN ---
if __name__ == "__main__":
    # Reset contador de estados
    State._id_counter = 0

    # Construir inductivamente el AFN para la expresión: (a | b)*
    nfa_a = ThompsonNFA.build_symbol('a')      # a
    nfa_b = ThompsonNFA.build_symbol('b')      # b
    nfa_ab = ThompsonNFA.build_union(nfa_a, nfa_b)  # (a | b)
    nfa_star = ThompsonNFA.build_kleene(nfa_ab)     # (a | b)*

    nfa_star.print_nfa()

    # Prueba de integración: AFN para a?
    # Reinicio el contador para que los estados de este autómata empiecen
    # desde q0 y no se mezclen con los de (a|b)*
    State._id_counter = 0

    nfa_a2 = ThompsonNFA.build_symbol('a')            # a   -> q0 --a--> q1
    nfa_opcional = ThompsonNFA.build_optional(nfa_a2)  # a?  -> agrega q2 (inicio) y q3 (final)

    print("\nPrueba con la expresión a?")
    nfa_opcional.print_nfa()

    # Lo que debería salir:
    #   q2 --(ε)--> q0   entra al sub-autómata para leer la a
    #   q2 --(ε)--> q3   se salta la a, así acepta la cadena vacía
    #   q0 --(a)--> q1   lee la a
    #   q1 --(ε)--> q3   después de leerla llega al final
    # Con esto el AFN acepta "" y "a", pero no "aa" porque no hay forma
    # de regresar a q0 después de leer la a.


=== AFN Thompson (Inicio: q6, Aceptación: q7) ===
  q6 --(ε)--> q7
  q6 --(ε)--> q4
  q4 --(ε)--> q0
  q4 --(ε)--> q2
  q2 --(b)--> q3
  q3 --(ε)--> q5
  q5 --(ε)--> q7
  q5 --(ε)--> q4
  q0 --(a)--> q1
  q1 --(ε)--> q5

Prueba con la expresión a?

=== AFN Thompson (Inicio: q2, Aceptación: q3) ===
  q2 --(ε)--> q0
  q2 --(ε)--> q3
  q0 --(a)--> q1
  q1 --(ε)--> q3


### Formato de Entrega / Portafolio de Evidencias
1. Trabajen en equipos de dos personas, entregando como reporte en el portafolio de evidencias:

*   El código fuente modificado con comentarios explicativos.

*   Capturas de pantalla o salidas de consola de las pruebas realizadas.

*   Respuesta conceptual a las preguntas de análisis lógico integradas en cada desafío.

2. Guarda este cuaderno con todas las salidas ejecutadas (`Archivo` -> `Guardar una copia en GitHub` / `PEREZ JUAN PSSB I Colab U3.ipynb`).

3. Registra en **Moodle** el enlace del cuaderno ejecutable en Colab y el de tu repositorio conteniendo la especificación formal del proyecto.